## This notebook uses df_match_features.csv only. Form, h2h, and FIFA rank are applied as adjustments in the simulation notebook, not as training features.

## Data I have:

- Strength signals (Elo, FIFA rank) -> **df_match_features.csv**
- Form signal (recent goal differential, per team) -> **df_form_2026.csv**
- Pair-specific history (h2h with sample-size guard) -> **df_h2h_2026.csv**
- Match context (tournament weight, neutral venue, confederation) -> **df_match_features.csv**

# Model Goal

- Poisson goals model: predict λ_home and λ_away (expected goals for each side), then derive outcome probabilities by sampling.

In [1]:
import pandas as pd
df = pd.read_csv('../data/processed/df_match_features.csv', parse_dates=['date'])
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

Shape: (49048, 16)

Columns:
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'home_elo_pre', 'away_elo_pre', 'elo_diff', 'tournament_weight', 'is_competitive', 'home_confederation', 'away_confederation']

Dtypes:
date                  datetime64[us]
home_team                        str
away_team                        str
home_score                   float64
away_score                   float64
tournament                       str
city                             str
country                          str
neutral                         bool
home_elo_pre                 float64
away_elo_pre                 float64
elo_diff                     float64
tournament_weight              int64
is_competitive                  bool
home_confederation               str
away_confederation               str
dtype: object

Missing values:
Series([], dtype: int64)


### What the model will actually use
Out of those 16 columns, here's what's what:

##### Identifiers (not features): date, home_team, away_team, city, country. These tell you which match it is, not anything predictive.

##### Targets: home_score, away_score. The things you're predicting. They can't be in the features.
##### Drop: tournament, is_competitive. tournament has too many unique values to one-hot encode sensibly (200+ tournaments). is_competitive is information you'll use to filter the training set, not as a feature.

Actual features to feed the model (7):

- home_elo_pre — home team strength
- away_elo_pre — away team strength
- elo_diff — useful for simpler models, redundant for tree models but harmless
- tournament_weight — match importance
- neutral — venue type (boolean)
- home_confederation — categorical (one-hot it)
- away_confederation — categorical (one-hot it)

That's a focused, principled feature set. Not 50 columns of noise.

In [2]:
print(df.shape)
df.head(2)

(49048, 16)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.0000,1500.0000,0.0000,1,False,UEFA,UEFA
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1502.8013,1497.1987,5.6026,1,False,UEFA,UEFA


In [3]:
df['is_competitive'].value_counts()   

is_competitive
True     30796
False    18252
Name: count, dtype: int64

In [4]:
# Use only competitive matches for training (remove friendlies and the cold-start NaN rows)
train_df = df[df['is_competitive']].copy()
print(f"Training on {len(train_df):,} competitive matches")

Training on 30,796 competitive matches


### Why two separate Poisson models
This is worth understanding before you write code. Football models predict goals as two independent Poisson distributions:

- Model A: predict λ_home (expected goals scored by home team)
- Model B: predict λ_away (expected goals scored by away team)

Both models use the same features, just with the target swapped. Then you sample from each Poisson to get score lines, and outcomes derive from there.

# Train test split

In [5]:
# Build the training set / Filter out before 2000
train_df = df[df['is_competitive'] & (df['date'] >= '2000-01-01')].copy()

train = train_df[train_df['date'] < '2024-01-01']
valid = train_df[train_df['date'] >= '2024-01-01']

print(f"Train: {len(train):,} matches ({train['date'].min().year}–{train['date'].max().year})")
print(f"Valid: {len(valid):,} matches ({valid['date'].min().year}–{valid['date'].max().year})")

Train: 14,820 matches (2000–2023)
Valid: 1,854 matches (2024–2026)


#### I picked 2000 to balance sample size against tactical era stability. Earlier would dilute the signal, later would limit data

# Fit 2 Poisson Models

In [6]:
import statsmodels.api as sm
import pandas as pd

# Define the feature columns
numeric_features = ['home_elo_pre', 'away_elo_pre', 'tournament_weight']
categorical_features = ['home_confederation', 'away_confederation']

def build_design_matrix(df):
    """Convert features into a numeric matrix the model can consume."""
    X = df[numeric_features].copy()
    # Neutral venue as a boolean → int
    X['neutral'] = df['neutral'].astype(int)
    # One-hot encode confederations (drop_first avoids the dummy-variable trap)
    cats = pd.get_dummies(df[categorical_features], drop_first=True).astype(int)
    X = pd.concat([X, cats], axis=1)
    # Add intercept column (required by statsmodels GLM)
    X = sm.add_constant(X, has_constant='add')
    return X

# Build design matrices
X_train = build_design_matrix(train)
X_valid = build_design_matrix(valid)

# Make sure validation has the same columns as train (handles any missing categories)
X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)

# Targets
y_home_train = train['home_score']
y_away_train = train['away_score']

# Fit Model A: expected home goals
model_home = sm.GLM(y_home_train, X_train, family=sm.families.Poisson()).fit()

# Fit Model B: expected away goals
model_away = sm.GLM(y_away_train, X_train, family=sm.families.Poisson()).fit()

print("=== Home goals model ===")
print(model_home.summary())
print("\n=== Away goals model ===")
print(model_away.summary())

=== Home goals model ===
                 Generalized Linear Model Regression Results                  
Dep. Variable:             home_score   No. Observations:                14820
Model:                            GLM   Df Residuals:                    14803
Model Family:                 Poisson   Df Model:                           16
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -23792.
Date:                Sun, 31 May 2026   Deviance:                       19633.
Time:                        22:08:16   Pearson chi2:                 1.97e+04
No. Iterations:                     5   Pseudo R-squ. (CS):             0.3786
Covariance Type:            nonrobust                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------